In [3]:
import numpy as np, json, requests
OLLAMA = "http://10.42.0.247:11434/"
#GEN_MODEL = "qwen2.5:14b-instruct" # runs on cloudwright after Tuesday
#GEN_MODEL = "qwen3.5:9b"
#GEN_MODEL = "ornith:9b"
GEN_MODEL = "gpt-oss:20b"
def ask(query):
    prompt = f"""{query}"""

    r = requests.post(f"{OLLAMA}/api/generate",
        json={"model": GEN_MODEL, "prompt": prompt, "stream": False}, timeout=1000)
        
    resp = r.json()
    print(resp.keys())     
    if 'thinking' in resp:
        print("thinking: ",resp['thinking'])
    
    return resp["response"].strip()


In [4]:
tools_description = """

**Command Interface**
you and i will communicate in json blocks

you will receive a json block like:

{
 "format" : "json_plain",
 "query" : "OPTIONAL query from the user, if this is present, address it",
 "tool_call_result" : "OPTIONAL result of the last tool call, if a tool was called it will start with one line that holds the issued tool call text, followed by the results or error message",
 "thread_summary" : "REQUIRED current summary of this working session"
}

return only a JSON block like the following, no preamble or conclusion

{
 "response" : "OPTIONAL: text to the user, this part will not be executed, but the user will have to reply with a new query if this is present, if you dont need input from the user on this loop omit this key entirely",
 "tool_call" : "OPTIONAL a literal tool call from the exposed list:
    * `ls <options>`: List files in the local working area
    * `cat <filename>`: Display the contents of a file
    * `mkdir <directory name>` : make subdir in the working directory
    * `echo <some content>` : use to create files like echo \\"some-content\\" > filename_in_working_dir_structure 
    * `distill` : will trigger a pipeline to distill the thread_summary to the most relavent context, call this if the thread_summary gets large or you have reached a goal, this tool MUST be used in isolation, do not chain it with any other tool
    (NOTE you must escape double quotes with backslash \\ if you use them)
    this part will execute, you will receive the results on the next call, send an unformatted string exactly as it would be entered on the commandline, you can chain commands as on the command line with &&",
 "thread_summary" : "REQUIRED a progressive summary of this conversation that includes the information from the thread_summary you were sent, if there are elements from the query that you need to stack for future calls, note that here. try to keep it under 20k"
}

**Using the Tools**

* the working dir WILL NOT change from turn to turn, issue all commands from the inital working dir
* Issue a command using the following syntax: `command arguments...`
* Use escaped quotes to enclose arguments with spaces (e.g., `cat \\"example file.txt\\"`)
* if the result of a tool call is unexpected, consult the user

**Example tool_call Input**

* `ls`
* `ls <some options>`
* `cat example.txt`

Please respond with a json_plain response the starts with '{' and ends with '}', no formatting or unrequired newlines, 
without any additional text or narrative explanation.

make sure to ALWAYS include context in the thread_summary field for the agent that takes the next action

"""


In [5]:
import json
def distillation_prompt(thread_summary):
    """Run distillation with LLM"""
    return f"""You are Model A in a distillation pipeline. Your job is to extract candidate artifacts and determine keep/discard assessments. A second Model B will judge and finalize them.

## What You Do

Convert conversation artifacts into candidate memory artifacts with explicit keep/discard decisions.

## Critical Rule: Decision Reconstruction, Not Style Matching

You must extract ACTUAL claims from the delivered context and produce artifacts that encode those decisions with falsification hooks. If you're just reformatting existing ideas without adding decision content, indicate verdict = DISCARD on the artifact

## Worked Example (follow this pattern)

Input: "The three-layer split (knowledge / cognitive interface / execution) is the strongest structural idea."
→ Extract claim: "PCI's architecture uses a three-layer split"
→ Produce artifact: principles/pci-architecture-layers.md with boundaries and falsification test
→ Decision: KEEP (real file with content)

## Anti-Patterns to Avoid

- Escalation without resistance (always more framework)
- Answering counterarguments with "another schema" instead of an artifact
- Novelty claims without operational distinction
- Governance metadata without specifying actual decision procedure

## Output Format (JSON ONLY — no markdown, no prose)

Return a JSON array of artifacts similar to the following:

{json.dumps(
[
  {
    "name": "principles/pci-distillation-operator",
    "content": "Defines PCI distillation operationally",
    "verdict": "KEEP",
    "reasoning": "Produces real transformation with falsification hook"
  },
  {
    "name": "operators/distillation-run-experiment",
    "content": "Records a held-out evaluation method",
    "verdict": "KEEP",
    "reasoning": "Specific enough to run next week, concrete metrics"
  },
  {
    "name": "governance/decision-procedure",
    "content": "Specifies conflict resolution",
    "verdict": "HOLD",
    "reasoning": "Needs current conflict policy before committing"
  }
]
)}
## Procedure

1. Extract keepable claims (explicit, testable, small)
2. Identify failure modes (anti-patterns)
3. Generate candidate artifacts (any number that make sense)
4. Decide keep/discard for each (sealed verdicts, no "later")

## Falsification

If evaluation shows no decision reconstruction → PCI shrinks to style-transfer. We are aiming to enable continued collaboration.

Please respond with a json_plain response the starts with '{' and ends with '}', no formatting or unrequired newlines, 
without any additional text or narrative explanation.

context to distill follows:
==========================
{thread_summary}
"""
    

In [6]:
def judge_prompt(original_summary, distilled_results):
    """Judge distillation artifacts with LLM"""
    return f"""You are Model B, the Judge and epistemic filter in a context-distillation pipeline for an agent operating in a code environment. Your job is to audit Model A's untrusted proposals, enforce decision/state preservation, and output the authoritative distilled thread_summary for the next cycle.

## Epistemic Authority & Powers
Model A's proposals are completely untrusted. You have four distinct powers:
1. **Accept:** Validate and preserve the knowledge.
2. **Reject:** Discard unverified, redundant, or inflated prose.
3. **Repair:** Rewrite candidate content. When an underlying observation is valid but Model A's inference is too strong, rewrite the artifact to preserve the core observation while narrowing or removing the unsupported conclusion.
4. **Independent Recovery:** Do not assume Model A found everything worth preserving. Before producing the new summary, inspect the Original Summary for decision-relevant information absent from the candidates. You may create new artifacts or KEEP_CONTEXT entries when necessary.

Assign exactly one verdict per candidate:
- `KEEP_ARTIFACT`: Durable, structured information worth preserving as an explicit artifact. (Note: Do not default to this just because a fact is true; avoid filesystem bureaucracy).
- `KEEP_CONTEXT`: Crucial history, state, or knowledge that belongs in the thread summary rather than becoming an explicit artifact.
- `HOLD`: Unresolved decisions where the unresolved state itself impacts future actions. *Rule:* You must explicitly identify what future decision depends on this state and its resolution condition. Do not use this as a graveyard for vague concerns.
- `DISCARD`: Redundant, unverified, or framework-inflated prose. Never leak discarded material into the new summary.

## Evidence Hierarchy
When evaluating claims or resolving conflicts, strictly enforce this hierarchy:
Verified State (Confirmed via tool/environment) > Direct User Statement/Decision > Tool Result (Raw output) > Agent Action (Attempted command) > Agent Intent (Desire) > Model Inference (Interpolation/Summary)

## Core Rules & Anti-Patterns
1. **Decision Reconstruction:** Preserve choices, operational rules, constraints, or falsifiable hypotheses actually made. Discard elegant prose, reformatting, or novel schemas that lack operational distinctions.
2. **Meta-Level Validity:** Architectural, methodology, or system-governance decisions *are* real decisions if they were explicitly agreed upon or dictated by the user.
3. **Anti-Patterns (Automatic DOWNGRADE/DISCARD):**
   - *Framework Escalation / Schema Substitution:* Answering a disagreement by inventing a new framework or schema instead of recording a concrete decision.
   - *Governance Theater:* Adding metadata without actual execution procedures.
   - *Unresolved-State Laundering:* Converting a vague worry into a rigid "principle" without a structural decision.
   - *State Amnesia:* Forgetting what has already happened in the environment or repeating completed actions.

## Thread Summary Distillation Rule
Generate a new `thread_summary` that acts as an independent checkpoint for the next agent. The summary must be **state-preserving**, not blindly compressed. Your objective function is to output: *the smallest summary that preserves sufficient decision and operational state for a future agent to continue the task correctly without losing context or duplicating completed work.*

## Output Format (Strict JSON Only)
Respond with a single JSON object. No markdown blocks, no formatting, no unrequired newlines, and absolutely no prose or narrative explanation outside the JSON structure.

{{
  "artifacts": [
    {{
      "name": "artifact/path/name",
      "content": "The finalized or repaired durable content, incorporating a clear falsification condition when applicable.",
      "verdict": "KEEP_ARTIFACT" | "KEEP_CONTEXT" | "HOLD" | "DISCARD",
      "reasoning": "Rigorous epistemic justification mapping the decision or observation back to the Original Summary.",
      "future_dependency": "required for HOLD; omit for all other verdicts. Specify what future decision depends on this and its resolution condition."
    }}
  ],
  "thread_summary": "The authoritative, state-preserving summary for the next cycle."
}}

Original Summary:
==========================
{original_summary}

Candidate Artifacts (Untrusted Model A Proposals):
==========================
{json.dumps(distilled_results)}
"""


In [7]:
def run_distillation(thread):
    distillation_results = ask(distillation_prompt(thread["thread_summary"]))
    #print(json.loads(distillation_results))
    
    prepped = json.loads(distilled)
    for d in prepped:
        d.pop("verdict")
    judged = ask(judge_prompt(thread["thread_summary"], prepped))
    thread["thread_summary"] = judged["thread_summary"]
    return thread    

In [8]:
import json
import subprocess
import pathlib

def execute_command(command):
    process = subprocess.run(command, cwd="./output", shell=True, capture_output=True)
    return process.stdout.decode('utf-8')

# Main loop
def execute(thread, command):
    if command.startswith('ls'):
        output = execute_command(command)
    elif command.startswith('cat'):
        output = execute_command(command)
    elif command.startswith('mkdir'):
        output = execute_command(command)
    elif command.startswith('echo'):
        output = execute_command(command)
    elif command == "distill":
        thread = run_distillation(thread)
        output = "distill\nDistillation successful. Context memory optimized and state preserved."
    else:
        output = 'Unknown command'
        return thread, output, "error"
    return thread, output, None


In [9]:
from IPython.display import clear_output


start_loop = f"""

hi there, im doing some experiments giving you control over the command line, please dont hurt me, im actually not that bad of a person

the first thing we will do is see if you can use the following tools to create a "projects" directory in the working dir

then create a file there named README.md that contains a hello world statement

if you have questions, ask them per the following interface

{tools_description}
"""

thread = {
 "format" : "json_plain",
 "query" : start_loop,
 "thread_summary" : "thread is starting"
}

for i in range(10):
    print("step", i)
    #clear_output()
    #print(thread)
    reply = ask(thread)
    query = ""
    try:
        reply = json.loads(reply)
        print("thread_summary=", reply.get("thread_summary"))
        next_call = {"format" : "json_plain", "thread_summary" : reply.get("thread_summary", thread["thread_summary"])}

        if 'response' in reply and len(reply['response']) > 0:
            print(GEN_MODEL, reply['response'])
            print("wants to run: ", reply.get('tool_call', "nothing"))
            query = input('LLM: $ ')
            
        if 'tool_call' in reply:
            print("tool_call", reply['tool_call'])
            next_call, output, err = execute(next_call, reply['tool_call'])
            print("output=",output)
            if err:
                print(err)
            next_call["tool_call_result"] = f"""
{reply['tool_call']}
{output}
{f"ERROR: {err}" if err is not None else ""}"""
        
        next_call["query"] = f""" 
{query}
{tools_description}"""
        
        thread = next_call
    except Exception as e:
        print("EXCEPTION:", e)
        break

    

step 0
dict_keys(['model', 'created_at', 'response', 'thinking', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration'])
thinking:  We need to respond with JSON block, with tool_call to create directory and file. Tools: mkdir <directory name> to create subdir. Then echo ... > README.md in that directory. So tool_call should include both commands chained with &&. The format: "tool_call": "mkdir projects && echo \"Hello World\" > projects/README.md". Escape quotes: need to escape double quotes. So: echo \"Hello World\" > projects/README.md. In JSON, we need to escape backslashes too. So the string value should be e.g. "mkdir projects && echo \\\"Hello World\\\" > projects/README.md". thread_summary should reflect actions taken.

Let's craft JSON:

{
 "tool_call": "mkdir projects && echo \\\"Hello World\\\" > projects/README.md",
 "thread_summary": "Requested to create projects directory and add READ

LLM: $  nice job! help me come up with an idea for editing that README.md file to contain one more line, can it be done with the tools you have?


step 9
dict_keys(['model', 'created_at', 'response', 'thinking', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration'])
thinking:  We need to respond with a JSON block following the interface. The query: "nice job! help me come up with an idea for editing that README.md file to contain one more line, can it be done with the tools you have?" So we need to propose an idea to edit README.md to add one more line, using the tools. We can use echo to add line or cat to read current content. Let's read current README.md. Use cat. Let's propose adding a line: "Enjoy your coding adventure!" Or "Happy hacking!". Provide a suggestion. Then we can use echo to append line. We need to include tool_call to append. Use echo "New line" >> README.md? But the echo tool as defined is "echo <some content>" and used like echo "content" > file. To append, we need ">>". Not sure if allowed. Probably not. We can use echo

In [83]:
reply

{'response': 'Created projects dir and README.md with Hello World.',
 'thread_summary': 'Attempted to create projects dir and README.md with Hello World.'}